# Amazon Bedrock AgentCore Policy - MCP 서버 타겟 정리

## 개요

이 노트북은 `01-Setup-MCP-Runtime-Gateway.ipynb`와 `02-Policy-Enforcement.ipynb`에서 생성한 모든 AWS 리소스를 삭제합니다.

### 삭제할 리소스

| 순서 | 리소스 | 설명 |
|------|--------|------|
| 1 | Policy Engine | Cedar 정책 엔진 (Gateway에서 분리 후 삭제) |
| 2 | MCP Target | Gateway에 연결된 MCP 서버 타겟 |
| 3 | AgentCore Gateway | MCP 프로토콜 엔드포인트 |
| 4 | OAuth Credential Provider | Identity (Outbound Auth용) |
| 5 | AgentCore Runtime | MCP 서버 호스팅 컨테이너 |
| 6 | Custom Claims Lambda | Cognito Pre Token Generation 트리거용 Lambda |
| 7 | Runtime Cognito | Runtime OAuth용 User Pool |
| 8 | Gateway Cognito | Gateway Inbound Auth용 User Pool |
| 9 | Config Files | 설정 파일 (선택) |

> **주의**: 이 노트북을 실행하면 모든 리소스가 **영구적으로 삭제**됩니다.

> **참고**: Gateway와 Gateway Cognito는 `01-Lambda-Target`과 공유됩니다. 이 노트북을 실행하면 Lambda 타겟 튜토리얼도 재설정이 필요합니다.

---

## Part 1: 환경 설정

In [19]:
import json
import sys
import time
from pathlib import Path

import boto3
from botocore.exceptions import ClientError

print("✓ Libraries loaded")

✓ Libraries loaded


### Step 1.1: 설정 파일 로드

In [20]:
# Load configuration files
runtime_config_path = Path.cwd() / "runtime_config.json"
cognito_config_path = Path.cwd() / "cognito_config.json"
gateway_cognito_config_path = Path.cwd() / "gateway_cognito_config.json"
lambda_target_gateway_config_path = Path.cwd().parent / "01-Lambda-Target" / "gateway_config.json"

RUNTIME_CONFIG = None
COGNITO_CONFIG = None
GATEWAY_COGNITO_CONFIG = None
GATEWAY_CONFIG = None

# Load Runtime config
if runtime_config_path.exists():
    with open(runtime_config_path, "r") as f:
        RUNTIME_CONFIG = json.load(f)
    print("✓ runtime_config.json loaded")
else:
    print("⚠️  runtime_config.json not found")

# Load Runtime Cognito config
if cognito_config_path.exists():
    with open(cognito_config_path, "r") as f:
        COGNITO_CONFIG = json.load(f)
    print("✓ cognito_config.json loaded")
else:
    print("⚠️  cognito_config.json not found")

# Load Gateway Cognito config
if gateway_cognito_config_path.exists():
    with open(gateway_cognito_config_path, "r") as f:
        GATEWAY_COGNITO_CONFIG = json.load(f)
    print("✓ gateway_cognito_config.json loaded")
else:
    print("⚠️  gateway_cognito_config.json not found")

# Load Gateway config from Lambda Target directory
if lambda_target_gateway_config_path.exists():
    with open(lambda_target_gateway_config_path, "r") as f:
        GATEWAY_CONFIG = json.load(f)
    print("✓ gateway_config.json loaded (from 01-Lambda-Target)")
else:
    print("⚠️  gateway_config.json not found")

✓ runtime_config.json loaded
✓ cognito_config.json loaded
✓ gateway_cognito_config.json loaded
✓ gateway_config.json loaded (from 01-Lambda-Target)


In [21]:
# Extract configuration values
if RUNTIME_CONFIG:
    region = RUNTIME_CONFIG.get("region", "us-east-1")
    runtime_id = RUNTIME_CONFIG.get("runtime_id")
    runtime_arn = RUNTIME_CONFIG.get("runtime_arn")
    gateway_id = RUNTIME_CONFIG.get("gateway_id")
    gateway_arn = RUNTIME_CONFIG.get("gateway_arn")
    target_id = RUNTIME_CONFIG.get("target_id")
    credential_provider_arn = RUNTIME_CONFIG.get("credential_provider_arn")
    credential_provider_name = RUNTIME_CONFIG.get("credential_provider_name")
    policy_engine_arn = RUNTIME_CONFIG.get("policy_engine_arn")
    policy_engine_id = policy_engine_arn.split("/")[-1] if policy_engine_arn else None
else:
    region = "us-east-1"
    runtime_id = None
    runtime_arn = None
    gateway_id = GATEWAY_CONFIG.get("gateway_id") if GATEWAY_CONFIG else None
    gateway_arn = GATEWAY_CONFIG.get("gateway_arn") if GATEWAY_CONFIG else None
    target_id = None
    credential_provider_arn = None
    credential_provider_name = None
    policy_engine_arn = GATEWAY_CONFIG.get("policy_engine_id") if GATEWAY_CONFIG else None
    policy_engine_id = policy_engine_arn

# Runtime Cognito (handle both "user_pool_id" and "pool_id" keys)
if COGNITO_CONFIG:
    runtime_user_pool_id = COGNITO_CONFIG.get("user_pool_id") or COGNITO_CONFIG.get("pool_id")
    runtime_client_id = COGNITO_CONFIG.get("client_id")
    runtime_scope = COGNITO_CONFIG.get("scope", "")
    runtime_domain = COGNITO_CONFIG.get("domain")
else:
    runtime_user_pool_id = None
    runtime_client_id = None
    runtime_scope = ""
    runtime_domain = None
runtime_resource_server_id = runtime_scope.split("/")[0] if runtime_scope else None

# Gateway Cognito (from gateway_config.json)
if GATEWAY_CONFIG:
    gateway_user_pool_id = GATEWAY_CONFIG.get("client_info", {}).get("user_pool_id")
    gateway_client_id = GATEWAY_CONFIG.get("client_info", {}).get("client_id")
    gateway_domain_prefix = GATEWAY_CONFIG.get("client_info", {}).get("domain_prefix")
    gateway_scope = GATEWAY_CONFIG.get("client_info", {}).get("scope", "")
    gateway_resource_server_id = gateway_scope.split("/")[0] if gateway_scope else "TestGateway"
else:
    gateway_user_pool_id = GATEWAY_COGNITO_CONFIG.get("user_pool_id") if GATEWAY_COGNITO_CONFIG else None
    gateway_client_id = None
    gateway_domain_prefix = None
    gateway_scope = ""
    gateway_resource_server_id = None

# Custom claims Lambda
custom_claims_lambda = f"cognito-custom-claims-{gateway_user_pool_id}" if gateway_user_pool_id else None

print("=" * 70)
print("🗑️  삭제될 리소스 목록")
print("=" * 70)

print("\n📦 Policy Engine:")
print(f"   - Policy Engine ID: {policy_engine_id or 'Not set'}")

print("\n🌐 Gateway:")
print(f"   - Gateway ID: {gateway_id or 'Not set'}")
print(f"   - MCP Target ID: {target_id or 'Not set'}")

print("\n🔑 Identity (OAuth Credential Provider):")
print(f"   - Name: {credential_provider_name or 'Not set'}")
print(f"   - ARN: {credential_provider_arn or 'Not set'}")

print("\n🖥️  AgentCore Runtime:")
print(f"   - Runtime ID: {runtime_id or 'Not set'}")
print(f"   - Runtime ARN: {runtime_arn or 'Not set'}")

print("\n⚡ Lambda Functions:")
print(f"   - Custom Claims Lambda: {custom_claims_lambda or 'Not set'}")

print("\n🔐 Runtime Cognito:")
print(f"   - User Pool ID: {runtime_user_pool_id or 'Not set'}")
print(f"   - App Client ID: {runtime_client_id or 'Not set'}")
print(f"   - Resource Server: {runtime_resource_server_id or 'Not set'}")
print(f"   - Domain: {runtime_domain or 'Not set'}")

print("\n🔐 Gateway Cognito:")
print(f"   - User Pool ID: {gateway_user_pool_id or 'Not set'}")
print(f"   - App Client ID: {gateway_client_id or 'Not set'}")
print(f"   - Resource Server: {gateway_resource_server_id or 'Not set'}")
print(f"   - Domain: {gateway_domain_prefix or 'Not set'}")

print("\n📄 Config Files:")
print(f"   - runtime_config.json")
print(f"   - cognito_config.json")
print(f"   - gateway_cognito_config.json")
print(f"   - ../01-Lambda-Target/gateway_config.json")

print("\n" + "=" * 70)
print(f"Region: {region}")
print("=" * 70)

🗑️  삭제될 리소스 목록

📦 Policy Engine:
   - Policy Engine ID: PolicyEngine_1766914563-29av1_xdkn

🌐 Gateway:
   - Gateway ID: testgwforpolicyengine-yoem6wqcpg
   - MCP Target ID: OUPPOJQJYF

🔑 Identity (OAuth Credential Provider):
   - Name: refund-mcp-server-identity
   - ARN: arn:aws:bedrock-agentcore:us-east-1:057716757052:token-vault/default/oauth2credentialprovider/refund-mcp-server-identity

🖥️  AgentCore Runtime:
   - Runtime ID: refund_mcp_server-d1slxjGm0X
   - Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:057716757052:runtime/refund_mcp_server-d1slxjGm0X

⚡ Lambda Functions:
   - Custom Claims Lambda: cognito-custom-claims-us-east-1_dJHfyS3hD

🔐 Runtime Cognito:
   - User Pool ID: us-east-1_cukznCUsr
   - App Client ID: 6ai7g352ofslmuhsl6s4tiftnd
   - Resource Server: refund-mcp
   - Domain: refund-mcp-cukzncusr

🔐 Gateway Cognito:
   - User Pool ID: us-east-1_dJHfyS3hD
   - App Client ID: 6a614nllco57q9ub82vjgglmu3
   - Resource Server: TestGateway
   - Domain: agentcore-2dbe94

### Step 1.2: AWS 클라이언트 초기화

In [22]:
session = boto3.Session(region_name=region)

sts_client = session.client("sts")
lambda_client = session.client("lambda")
cognito_client = session.client("cognito-idp")
agentcore_client = session.client("bedrock-agentcore-control")

ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

print("✓ AWS clients initialized")
print(f"  Account ID: {ACCOUNT_ID}")
print(f"  Region: {region}")

✓ AWS clients initialized
  Account ID: 057716757052
  Region: us-east-1


---

## Part 2: Policy Engine 정리

Policy Engine을 Gateway에서 분리하고 삭제합니다.

### Step 2.1: Gateway에서 Policy Engine 분리

In [23]:
if gateway_id and policy_engine_id:
    print("=" * 70)
    print("Step 2.1: Detaching Policy Engine from Gateway")
    print("=" * 70)
    
    try:
        # Get current gateway configuration
        gateway = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)
        
        # Check if policy engine is attached (correct field: policyEngineConfiguration)
        pe_config = gateway.get("policyEngineConfiguration", {})
        
        if pe_config and pe_config.get("arn"):
            print(f"  Current Policy Engine: {pe_config.get('arn')}")
            print(f"  Mode: {pe_config.get('mode', 'N/A')}")
            
            # Update gateway WITHOUT policyEngineConfiguration parameter to detach
            # (omitting the parameter entirely removes the policy engine)
            agentcore_client.update_gateway(
                gatewayIdentifier=gateway_id,
                name=gateway.get("name"),
                roleArn=gateway.get("roleArn"),
                protocolType=gateway.get("protocolType", "MCP"),
                authorizerType=gateway.get("authorizerType", "CUSTOM_JWT"),
                authorizerConfiguration=gateway.get("authorizerConfiguration"),
            )
            print(f"✓ Policy Engine detached from Gateway")
            
            # Wait for gateway to be ready
            print("  Waiting for Gateway to be ready...")
            time.sleep(5)
        else:
            print("  Policy Engine was not attached to Gateway")
            
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Gateway {gateway_id} not found (may already be deleted)")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  No Gateway ID or Policy Engine ID found")

Step 2.1: Detaching Policy Engine from Gateway
  Gateway testgwforpolicyengine-yoem6wqcpg not found (may already be deleted)


### Step 2.2: Policy Engine 내 정책 삭제

In [24]:
if policy_engine_id:
    print("=" * 70)
    print("Step 2.2: Deleting all policies in Policy Engine")
    print("=" * 70)
    
    try:
        # List all policies
        paginator = agentcore_client.get_paginator("list_policies")
        deleted_count = 0
        
        for page in paginator.paginate(policyEngineId=policy_engine_id):
            for policy in page.get("policies", []):
                pid = policy["policyId"]
                try:
                    agentcore_client.delete_policy(
                        policyEngineId=policy_engine_id,
                        policyId=pid
                    )
                    print(f"  ✓ Deleted policy: {pid}")
                    deleted_count += 1
                except ClientError as e:
                    print(f"  ✗ Failed to delete policy {pid}: {e}")
        
        if deleted_count == 0:
            print("  No policies found")
        else:
            print(f"\n✓ Deleted {deleted_count} policies")
            
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Policy Engine {policy_engine_id} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  No Policy Engine ID found")

Step 2.2: Deleting all policies in Policy Engine
  Policy Engine PolicyEngine_1766914563-29av1_xdkn not found


### Step 2.3: Policy Engine 삭제

In [25]:
if policy_engine_id:
    print("=" * 70)
    print("Step 2.3: Deleting Policy Engine")
    print("=" * 70)
    
    try:
        agentcore_client.delete_policy_engine(
            policyEngineId=policy_engine_id
        )
        print(f"✓ Policy Engine deleted: {policy_engine_id}")
        
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Policy Engine {policy_engine_id} not found (may already be deleted)")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  No Policy Engine ID found")

Step 2.3: Deleting Policy Engine
  Policy Engine PolicyEngine_1766914563-29av1_xdkn not found (may already be deleted)


---

## Part 3: Gateway 정리

Gateway Target을 삭제한 후 Gateway를 삭제합니다.

### Step 3.1: Gateway Target 삭제 (모든 타겟)

In [26]:
if gateway_id:
    print("=" * 70)
    print("Step 3.1: Deleting Gateway Targets")
    print("=" * 70)
    
    try:
        # List all targets (response key is "items", not "targets")
        response = agentcore_client.list_gateway_targets(gatewayIdentifier=gateway_id)
        targets = response.get("items", [])
        
        if targets:
            for target in targets:
                tid = target["targetId"]
                target_name = target.get("name", "Unknown")
                
                try:
                    agentcore_client.delete_gateway_target(
                        gatewayIdentifier=gateway_id,
                        targetId=tid
                    )
                    print(f"  ✓ Deleted target: {target_name} ({tid})")
                    
                    # Wait for target deletion
                    time.sleep(2)
                    
                except ClientError as e:
                    print(f"  ✗ Failed to delete target {target_name}: {e}")
        else:
            print("  No targets found")
            
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Gateway {gateway_id} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  No Gateway ID found")

Step 3.1: Deleting Gateway Targets
  Gateway testgwforpolicyengine-yoem6wqcpg not found


### Step 3.2: Gateway 삭제

In [27]:
if gateway_id:
    print("=" * 70)
    print("Step 3.2: Deleting Gateway")
    print("=" * 70)
    
    try:
        agentcore_client.delete_gateway(gatewayIdentifier=gateway_id)
        print(f"✓ Gateway deletion initiated: {gateway_id}")
        
        # Wait for gateway deletion
        print("  Waiting for Gateway to be deleted...")
        for i in range(30):
            try:
                gateway = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)
                status = gateway.get("status", "UNKNOWN")
                print(f"  Status: {status}")
                time.sleep(5)
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    print("✓ Gateway deleted successfully")
                    break
                else:
                    raise
        else:
            print("⚠️  Gateway deletion is taking longer than expected")
            
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Gateway {gateway_id} not found (may already be deleted)")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  No Gateway ID found")

Step 3.2: Deleting Gateway
  Gateway testgwforpolicyengine-yoem6wqcpg not found (may already be deleted)


---

## Part 4: Identity (OAuth Credential Provider) 정리

In [28]:
print("=" * 70)
print("Step 4: Deleting OAuth Credential Provider (Identity)")
print("=" * 70)

if credential_provider_name:
    try:
        # Find credential provider by name (response key is "credentialProviders")
        providers = agentcore_client.list_oauth2_credential_providers().get("credentialProviders", [])
        
        provider = next(
            (p for p in providers if p["name"] == credential_provider_name),
            None
        )
        
        if provider:
            agentcore_client.delete_oauth2_credential_provider(
                name=credential_provider_name
            )
            print(f"✓ Deleted Credential Provider: {credential_provider_name}")
        else:
            print(f"  Credential Provider {credential_provider_name} not found")
            
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Credential Provider {credential_provider_name} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  No Credential Provider name found in config")

Step 4: Deleting OAuth Credential Provider (Identity)
  Credential Provider refund-mcp-server-identity not found


---

## Part 5: AgentCore Runtime 정리
- 약 5분 정도 소요될 수 있습니다.

In [29]:
print("=" * 70)
print("Step 5: Deleting AgentCore Runtime")
print("=" * 70)

if runtime_id:
    try:
        agentcore_client.delete_agent_runtime(agentRuntimeId=runtime_id)
        print(f"✓ Runtime deletion initiated: {runtime_id}")
        
        # Wait for runtime deletion
        print("  Waiting for Runtime to be deleted...")
        for i in range(60):  # Runtime deletion can take longer
            try:
                runtime = agentcore_client.get_agent_runtime(agentRuntimeId=runtime_id)
                status = runtime.get("status", "UNKNOWN")
                print(f"  Status: {status}")
                time.sleep(10)
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    print("✓ Runtime deleted successfully")
                    break
                else:
                    raise
        else:
            print("⚠️  Runtime deletion is taking longer than expected")
            print("   You may need to check the AWS console.")
            
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Runtime {runtime_id} not found (may already be deleted)")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  No Runtime ID found in config")

Step 5: Deleting AgentCore Runtime
  Runtime refund_mcp_server-d1slxjGm0X not found (may already be deleted)


---

## Part 6: Lambda 함수 정리

### Step 6.1: Gateway Cognito Lambda Trigger 제거

In [30]:
if gateway_user_pool_id:
    print("=" * 70)
    print("Step 6.1: Removing Gateway Cognito Lambda Trigger")
    print("=" * 70)
    
    try:
        # Get current User Pool configuration
        user_pool = cognito_client.describe_user_pool(UserPoolId=gateway_user_pool_id)
        lambda_config = user_pool.get("UserPool", {}).get("LambdaConfig", {})
        
        if lambda_config.get("PreTokenGenerationConfig") or lambda_config.get("PreTokenGeneration"):
            # Remove Lambda trigger
            cognito_client.update_user_pool(
                UserPoolId=gateway_user_pool_id,
                LambdaConfig={}
            )
            print("✓ Lambda trigger removed from Gateway User Pool")
        else:
            print("  No Lambda trigger configured")
            
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  User Pool {gateway_user_pool_id} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  No Gateway User Pool ID found")

Step 6.1: Removing Gateway Cognito Lambda Trigger


  No Lambda trigger configured


### Step 6.2: Custom Claims Lambda 함수 삭제

In [31]:
print("=" * 70)
print("Step 6.2: Deleting Custom Claims Lambda Function")
print("=" * 70)

if custom_claims_lambda:
    try:
        lambda_client.delete_function(FunctionName=custom_claims_lambda)
        print(f"✓ Deleted Lambda function: {custom_claims_lambda}")
        
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Lambda function {custom_claims_lambda} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  No Custom Claims Lambda function name found")

Step 6.2: Deleting Custom Claims Lambda Function
  Lambda function cognito-custom-claims-us-east-1_dJHfyS3hD not found


### Step 6.3: Refund Lambda 함수 삭제 (01-Lambda-Target용)

In [32]:
print("=" * 70)
print("Step 6.3: Deleting Refund Lambda Function")
print("=" * 70)

# Lambda function name from gateway_config.json
lambda_arn = GATEWAY_CONFIG.get("lambda_arn") if GATEWAY_CONFIG else None

if lambda_arn:
    lambda_name = lambda_arn.split(":")[-1]
    
    try:
        lambda_client.delete_function(FunctionName=lambda_name)
        print(f"✓ Deleted Lambda function: {lambda_name}")
        
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Lambda function {lambda_name} not found")
        else:
            print(f"  Error: {e}")
else:
    # Try default name
    try:
        lambda_client.delete_function(FunctionName="RefundLambda")
        print("✓ Deleted Lambda function: RefundLambda")
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print("  Lambda function RefundLambda not found")
        else:
            print(f"  Error: {e}")

Step 6.3: Deleting Refund Lambda Function
  Lambda function RefundLambda not found


---

## Part 7: Runtime Cognito 정리

Runtime OAuth에 사용된 Cognito 리소스들을 삭제합니다.

### Step 7.1: Runtime Cognito App Client 삭제

In [33]:
if runtime_user_pool_id and runtime_client_id:
    print("=" * 70)
    print("Step 7.1: Deleting Runtime Cognito App Client")
    print("=" * 70)
    
    try:
        cognito_client.delete_user_pool_client(
            UserPoolId=runtime_user_pool_id,
            ClientId=runtime_client_id
        )
        print(f"✓ Deleted App Client: {runtime_client_id}")
        
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  App Client {runtime_client_id} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  Missing Runtime User Pool ID or Client ID")

Step 7.1: Deleting Runtime Cognito App Client


✓ Deleted App Client: 6ai7g352ofslmuhsl6s4tiftnd


### Step 7.2: Runtime Cognito Resource Server 삭제

In [35]:
if runtime_user_pool_id and runtime_resource_server_id:
    print("=" * 70)
    print("Step 7.2: Deleting Runtime Cognito Resource Server")
    print("=" * 70)
    
    try:
        cognito_client.delete_resource_server(
            UserPoolId=runtime_user_pool_id,
            Identifier=runtime_resource_server_id
        )
        print(f"✓ Deleted Resource Server: {runtime_resource_server_id}")
        
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Resource Server {runtime_resource_server_id} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  Missing Runtime User Pool ID or Resource Server ID")

Step 7.2: Deleting Runtime Cognito Resource Server
  Resource Server refund-mcp not found


### Step 7.3: Runtime Cognito Domain 삭제

In [36]:
if runtime_user_pool_id:
    print("=" * 70)
    print("Step 7.3: Deleting Runtime Cognito Domain")
    print("=" * 70)
    
    try:
        # Use domain from config if available, otherwise fetch from user pool
        domain = runtime_domain
        if not domain:
            user_pool = cognito_client.describe_user_pool(UserPoolId=runtime_user_pool_id)
            domain = user_pool.get("UserPool", {}).get("Domain")
        
        if domain:
            cognito_client.delete_user_pool_domain(
                Domain=domain,
                UserPoolId=runtime_user_pool_id
            )
            print(f"✓ Deleted Domain: {domain}")
        else:
            print("  No domain configured")
            
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  User Pool {runtime_user_pool_id} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  No Runtime User Pool ID found")

Step 7.3: Deleting Runtime Cognito Domain
✓ Deleted Domain: refund-mcp-cukzncusr


### Step 7.4: Runtime Cognito User Pool 삭제

In [37]:
if runtime_user_pool_id:
    print("=" * 70)
    print("Step 7.4: Deleting Runtime Cognito User Pool")
    print("=" * 70)
    
    try:
        cognito_client.delete_user_pool(UserPoolId=runtime_user_pool_id)
        print(f"✓ Deleted User Pool: {runtime_user_pool_id}")
        
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  User Pool {runtime_user_pool_id} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  No Runtime User Pool ID found")

Step 7.4: Deleting Runtime Cognito User Pool
✓ Deleted User Pool: us-east-1_cukznCUsr


---

## Part 8: Gateway Cognito 정리

Gateway Inbound Auth에 사용된 Cognito 리소스들을 삭제합니다.

> **참고**: 이 리소스들은 `01-Lambda-Target`과 공유됩니다.

### Step 8.1: Gateway Cognito App Client 삭제

In [38]:
if gateway_user_pool_id and gateway_client_id:
    print("=" * 70)
    print("Step 8.1: Deleting Gateway Cognito App Client")
    print("=" * 70)
    
    try:
        cognito_client.delete_user_pool_client(
            UserPoolId=gateway_user_pool_id,
            ClientId=gateway_client_id
        )
        print(f"✓ Deleted App Client: {gateway_client_id}")
        
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  App Client {gateway_client_id} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  Missing Gateway User Pool ID or Client ID")

Step 8.1: Deleting Gateway Cognito App Client
✓ Deleted App Client: 6a614nllco57q9ub82vjgglmu3


### Step 8.2: Gateway Cognito Resource Server 삭제

In [39]:
if gateway_user_pool_id and gateway_resource_server_id:
    print("=" * 70)
    print("Step 8.2: Deleting Gateway Cognito Resource Server")
    print("=" * 70)
    
    try:
        cognito_client.delete_resource_server(
            UserPoolId=gateway_user_pool_id,
            Identifier=gateway_resource_server_id
        )
        print(f"✓ Deleted Resource Server: {gateway_resource_server_id}")
        
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Resource Server {gateway_resource_server_id} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  Missing Gateway User Pool ID or Resource Server ID")

Step 8.2: Deleting Gateway Cognito Resource Server
✓ Deleted Resource Server: TestGateway


### Step 8.3: Gateway Cognito Domain 삭제

In [40]:
if gateway_user_pool_id and gateway_domain_prefix:
    print("=" * 70)
    print("Step 8.3: Deleting Gateway Cognito Domain")
    print("=" * 70)
    
    try:
        cognito_client.delete_user_pool_domain(
            Domain=gateway_domain_prefix,
            UserPoolId=gateway_user_pool_id
        )
        print(f"✓ Deleted Domain: {gateway_domain_prefix}")
        
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Domain {gateway_domain_prefix} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  Missing Gateway User Pool ID or Domain Prefix")

Step 8.3: Deleting Gateway Cognito Domain
✓ Deleted Domain: agentcore-2dbe941b


### Step 8.4: Gateway Cognito User Pool 삭제

In [41]:
if gateway_user_pool_id:
    print("=" * 70)
    print("Step 8.4: Deleting Gateway Cognito User Pool")
    print("=" * 70)
    
    try:
        cognito_client.delete_user_pool(UserPoolId=gateway_user_pool_id)
        print(f"✓ Deleted User Pool: {gateway_user_pool_id}")
        
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  User Pool {gateway_user_pool_id} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  No Gateway User Pool ID found")

Step 8.4: Deleting Gateway Cognito User Pool
✓ Deleted User Pool: us-east-1_dJHfyS3hD


---

## Part 9: 설정 파일 정리

In [42]:
print("=" * 70)
print("Step 9: Deleting Configuration Files")
print("=" * 70)

config_files = [
    Path.cwd() / "runtime_config.json",
    Path.cwd() / "cognito_config.json",
    Path.cwd() / "gateway_cognito_config.json",
    Path.cwd().parent / "01-Lambda-Target" / "gateway_config.json",
]

for config_file in config_files:
    if config_file.exists():
        config_file.unlink()
        print(f"✓ Deleted: {config_file}")
    else:
        print(f"  {config_file.name} not found")

print("\n✓ Configuration files cleaned up")

Step 9: Deleting Configuration Files
✓ Deleted: /home/ubuntu/amazon-bedrock-agentcore-policy-tutorials/02-MCP-Server-Target/runtime_config.json
✓ Deleted: /home/ubuntu/amazon-bedrock-agentcore-policy-tutorials/02-MCP-Server-Target/cognito_config.json
✓ Deleted: /home/ubuntu/amazon-bedrock-agentcore-policy-tutorials/02-MCP-Server-Target/gateway_cognito_config.json
✓ Deleted: /home/ubuntu/amazon-bedrock-agentcore-policy-tutorials/01-Lambda-Target/gateway_config.json

✓ Configuration files cleaned up


---

## 결론

### 삭제된 리소스

✅ Policy Engine 및 정책  
✅ Gateway Target (MCP + Lambda)  
✅ AgentCore Gateway  
✅ OAuth Credential Provider (Identity)  
✅ AgentCore Runtime  
✅ Custom Claims Lambda 함수  
✅ Refund Lambda 함수  
✅ Runtime Cognito (User Pool, App Client, Resource Server, Domain)  
✅ Gateway Cognito (User Pool, App Client, Resource Server, Domain)  
✅ 설정 파일  

### 다시 시작하려면

**MCP 서버 타겟 튜토리얼:**
1. `01-Setup-MCP-Runtime-Gateway.ipynb`를 다시 실행하여 모든 리소스를 재생성하세요.

**Lambda 타겟 튜토리얼:**
1. `../01-Lambda-Target/01-Setup-Gateway-Lambda.ipynb`를 다시 실행하여 Gateway를 재생성하세요.